# Titanic 16: CatBoost RandomizedSearchCV

## 가설

Titanic 15는 Feature Engineering으로 Private AUC 0.91468을 기록했지만 CatBoost 하이퍼파라미터는 기본값에 가깝다. Feature를 절대 변경하지 않고 depth, learning rate, iterations, L2 regularization, split randomness, sampling 비율, numeric border 수를 조절하면 과적합을 줄이고 ROC-AUC ranking을 개선할 가능성이 있다. Accuracy나 threshold=0.5가 아니라 positive class 확률의 순위 성능을 기준으로 선택한다.

RandomizedSearchCV의 best_score_는 후보 선택에 사용된 점수이므로 독립적인 일반화 성능 추정치가 아니다.


In [1]:
from pathlib import Path
import json, warnings
import numpy as np, pandas as pd
from IPython.display import display
from scipy.stats import randint, loguniform, uniform
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
from common.interaction_experiments import FeaturePreprocessor, baseline_parameters
warnings.filterwarnings('ignore')
SEED, TARGET, ID = 42, 'survived', 'passengerid'
FEATURES = ['GenderClass','GenderIsChild','ClassIsChild','AgeBand']
train = pd.read_csv('csv/train.csv'); test = pd.read_csv('csv/test.csv'); template = pd.read_csv('csv/submission.csv')
X = train.drop(columns=[TARGET,ID]); y = train[TARGET].astype('int8'); X_test = test.drop(columns=[ID])
assert X.columns.equals(X_test.columns)
BASE = baseline_parameters().copy(); BASE.pop('cat_features', None); BASE.update({'allow_writing_files':False,'thread_count':1})
print(train.shape, test.shape, BASE)


(916, 12) (393, 11) {'verbose': 0, 'random_state': 42, 'allow_writing_files': False, 'thread_count': 1}


In [2]:
class Titanic16Transformer(BaseEstimator, TransformerMixin):
    # This transformer is fit inside each CV fold; no full-data preprocessing occurs before CV.
    def __init__(self, additions=tuple(FEATURES)): self.additions = tuple(additions)
    def fit(self, X, y=None):
        self.prep_ = FeaturePreprocessor(self.additions)
        self.prep_.fit_missing(X)
        self.prep_.fit_encoding(self.prep_.feature_frame(X, self.prep_.transform_missing(X)))
        return self
    def transform(self, X): return self.prep_.transform(X)

def pipe(params): return Pipeline([('prep', Titanic16Transformer()), ('model', CatBoostClassifier(**params))])
def proba(model, X): return model.predict_proba(X)[:, list(model.classes_).index(1)]
def metrics(model, xa, ya, xb, yb):
    pa, pb = proba(model, xa), proba(model, xb)
    aa, ab = roc_auc_score(ya,pa), roc_auc_score(yb,pb)
    return {'train_auc':aa,'validation_auc':ab,'gap':aa-ab}
def fixed_cv(params):
    skf=StratifiedKFold(5,shuffle=True,random_state=SEED); rows=[]; oof=np.empty(len(X))
    for fold,(tr,va) in enumerate(skf.split(X,y),1):
        m=pipe(params).fit(X.iloc[tr],y.iloc[tr]); p=proba(m,X.iloc[va]); oof[va]=p
        rows.append({'fold':fold,'auc':roc_auc_score(y.iloc[va],p)}); print(fold,rows[-1]['auc'])
    return pd.DataFrame(rows),oof
tr,va=train_test_split(train,test_size=.25,stratify=y,random_state=SEED)
Xtr,ytr=tr.drop(columns=[TARGET,ID]),tr[TARGET].astype('int8'); Xva,yva=va.drop(columns=[TARGET,ID]),va[TARGET].astype('int8')
assert len(tr)==687 and len(va)==229


## Titanic 15 Champion 재현

결측치 처리, Age 처리, AgeBand, FamilySize, IsAlone, OneHotEncoder(handle_unknown="ignore"), Scaling/SMOTE 미사용, passengerid 제외, seed 42와 positive class predict_proba를 고정한다. CV 각 fold에서 raw train에만 preprocessing을 fit한다.


In [3]:
champion=pipe(BASE).fit(Xtr,ytr); ch_hold=metrics(champion,Xtr,ytr,Xva,yva)
ch_folds,ch_oof=fixed_cv(BASE); ch_mean=ch_folds.auc.mean(); ch_std=ch_folds.auc.std(ddof=1); ch_oof_auc=roc_auc_score(y,ch_oof)
print('Titanic 15 Champion',ch_hold,'CV Mean',ch_mean,'CV Std',ch_std,'OOF AUC',ch_oof_auc)


1 0.9010651629072682


2 0.9237859140605136


3 0.8797991355199594


4 0.9121535723366387


5 0.9145690312738368
Titanic 15 Champion {'train_auc': 0.965299045217078, 'validation_auc': 0.9036428687591479, 'gap': 0.06165617645793009} CV Mean 0.9062745632196434 CV Std 0.016867827959369617 OOF AUC 0.9042465267214278


## RandomizedSearchCV

loss_function=Logloss, bootstrap_type=MVS, random_seed=42, verbose=0, allow_writing_files=False는 고정한다. bootstrap_type을 탐색하지 않는 이유는 Bayesian과 MVS/Bernoulli의 sampling 파라미터 조합이 달라 invalid combination을 만들 수 있기 때문이다. RandomizedSearchCV n_jobs=-1과 CatBoost nested parallelism 충돌을 피하려고 thread_count=1을 사용한다.


In [4]:
space={'model__iterations':randint(500,1501),'model__depth':randint(4,9),'model__learning_rate':loguniform(.005,.08),'model__l2_leaf_reg':loguniform(1.,20.),'model__random_strength':loguniform(.1,5.),'model__subsample':uniform(.65,.30),'model__border_count':[32,64,128,254]}
search=RandomizedSearchCV(pipe(BASE),space,n_iter=30,scoring='roc_auc',cv=StratifiedKFold(5,shuffle=True,random_state=SEED),random_state=SEED,n_jobs=-1,verbose=1,refit=True,return_train_score=True)
search.fit(X,y); print(search.best_params_); print(search.best_score_); print(search.best_index_)
r=pd.DataFrame(search.cv_results_); r['rank']=r.rank_test_score; r['train_auc']=r.mean_train_score; r['cv_auc']=r.mean_test_score; r['cv_std']=r.std_test_score; r['gap']=r.train_auc-r.cv_auc
for p in ['iterations','depth','learning_rate','l2_leaf_reg','random_strength','subsample','border_count']: r[p]=r['param_model__'+p].astype(float)
display(r.sort_values('rank_test_score')[['rank','iterations','depth','learning_rate','l2_leaf_reg','random_strength','subsample','border_count','train_auc','cv_auc','cv_std','gap','params']].head(10))
Path('results').mkdir(exist_ok=True); out=Path('results/titanic_16_randomized_search_results.csv'); assert not out.exists(); r.to_csv(out,index=False)


Fitting 5 folds for each of 30 candidates, totalling 150 fits


{'model__border_count': 128, 'model__depth': 4, 'model__iterations': 817, 'model__l2_leaf_reg': np.float64(1.9452208847287389), 'model__learning_rate': np.float64(0.006971115660907851), 'model__random_strength': np.float64(0.3746261155838959), 'model__subsample': np.float64(0.9328729111737557)}
0.9104809124259926
22


,rank,iterations,depth,learning_rate,l2_leaf_reg,random_strength,subsample,border_count,train_auc,cv_auc,cv_std,gap,params
22,1,817.0,4.0,0.006971,1.945221,0.374626,0.932873,128.0,0.948592,0.910481,0.015477,0.038111,"{'model__border_count': 128, 'model__depth': 4..."
7,2,705.0,6.0,0.008287,3.226871,1.920155,0.777547,128.0,0.948800,0.909173,0.012931,0.039627,"{'model__border_count': 128, 'model__depth': 6..."
26,3,938.0,7.0,0.009663,9.792418,1.726703,0.760335,254.0,0.954679,0.907221,0.014807,0.047458,"{'model__border_count': 254, 'model__depth': 7..."
14,4,532.0,5.0,0.011842,1.209738,0.356843,0.868882,254.0,0.962758,0.907001,0.018388,0.055757,"{'model__border_count': 254, 'model__depth': 5..."
15,5,1046.0,6.0,0.006966,4.114962,1.628477,0.878236,254.0,0.953397,0.906959,0.014956,0.046438,"{'model__border_count': 254, 'model__depth': 6..."
19,6,1056.0,5.0,0.006699,2.424534,0.596539,0.715532,64.0,0.959370,0.906691,0.016279,0.052679,"{'model__border_count': 64, 'model__depth': 5,..."
18,7,1040.0,6.0,0.006190,1.984601,0.310655,0.698366,128.0,0.968869,0.906284,0.017401,0.062584,"{'model__border_count': 128, 'model__depth': 6..."
16,8,892.0,6.0,0.006800,8.406084,0.557724,0.710516,64.0,0.953711,0.905943,0.015852,0.047768,"{'model__border_count': 64, 'model__depth': 6,..."
9,9,769.0,7.0,0.012364,8.834921,0.931466,0.806250,64.0,0.963399,0.904716,0.016680,0.058684,"{'model__border_count': 64, 'model__depth': 7,..."
1,10,958.0,6.0,0.026472,13.394335,1.595857,0.656175,128.0,0.976864,0.901976,0.019332,0.074888,"{'model__border_count': 128, 'model__depth': 6..."


In [5]:
BEST=BASE.copy(); BEST.update({k.replace('model__',''):v for k,v in search.best_params_.items()})
tuned=pipe(BEST).fit(Xtr,ytr); tu_hold=metrics(tuned,Xtr,ytr,Xva,yva); tu_folds,tu_oof=fixed_cv(BEST); tu_mean=tu_folds.auc.mean(); tu_std=tu_folds.auc.std(ddof=1); tu_oof_auc=roc_auc_score(y,tu_oof)
folds=ch_folds.rename(columns={'auc':'Champion AUC'}).merge(tu_folds.rename(columns={'auc':'Tuned AUC'}),on='fold'); folds['Delta']=folds['Tuned AUC']-folds['Champion AUC']; display(folds)
comparison=pd.DataFrame([['Train AUC',ch_hold['train_auc'],tu_hold['train_auc']],['Validation AUC',ch_hold['validation_auc'],tu_hold['validation_auc']],['Gap',ch_hold['gap'],tu_hold['gap']],['CV Mean',ch_mean,tu_mean],['CV Std',ch_std,tu_std],['OOF AUC',ch_oof_auc,tu_oof_auc]],columns=['Metric','Titanic 15','Titanic 16']); comparison['Delta']=comparison['Titanic 16']-comparison['Titanic 15']; display(comparison)
e1=Titanic16Transformer().fit(X).transform(X); e2=Titanic16Transformer().fit(X).transform(X); assert e1.columns.equals(e2.columns); print('Titanic 15 encoded feature count:',e1.shape[1]); print('Titanic 16 encoded feature count:',e2.shape[1]); print('Columns identical:',e1.columns.equals(e2.columns))


1 0.8972431077694236


2 0.9351004322400204


3 0.8909865242817188


4 0.9132341723874904


5 0.9158403254513094


,fold,Champion AUC,Tuned AUC,Delta
0,1,0.901065,0.897243,-0.003822
1,2,0.923786,0.935100,0.011315
2,3,0.879799,0.890987,0.011187
3,4,0.912154,0.913234,0.001081
4,5,0.914569,0.915840,0.001271


,Metric,Titanic 15,Titanic 16,Delta
0,Train AUC,0.965299,0.952909,-0.012390
1,Validation AUC,0.903643,0.900065,-0.003578
2,Gap,0.061656,0.052844,-0.008812
3,CV Mean,0.906275,0.910481,0.004206
4,CV Std,0.016868,0.017304,0.000436
5,OOF AUC,0.904247,0.907781,0.003534


Titanic 15 encoded feature count: 33
Titanic 16 encoded feature count: 33
Columns identical: True


## 전체 Train 재학습, Submission, Prediction 비교

Best parameters로 전체 train에 preprocessing을 새로 fit하고 test는 transform만 한다. `predict()`는 사용하지 않고 positive class `predict_proba()`를 사용한다. 0.0001 수준 차이는 사실상 동률로 보류한다.


In [6]:
final=pipe(BEST).fit(X,y); pred=proba(final,X_test); assert np.isfinite(pred).all() and ((pred>=0)&(pred<=1)).all()
sub=Path('submission/titanic_result_16_randomized_search.csv'); assert not sub.exists(); assert template[ID].tolist()==test[ID].tolist(); result=template.copy(); result[TARGET]=pred; result.to_csv(sub,index=False); saved=pd.read_csv(sub); assert len(saved)==len(test) and saved.columns.tolist()==[ID,TARGET] and saved[TARGET].between(0,1).all()
from scipy.stats import pearsonr,spearmanr
old=pd.read_csv('submission/titanic_result_15.csv')[TARGET].to_numpy(); new=saved[TARGET].to_numpy(); changes=int(np.sum((old>=.5)!=(new>=.5))); pred_cmp={'mean_absolute_prediction_difference':float(np.mean(abs(old-new))),'max_absolute_prediction_difference':float(np.max(abs(old-new))),'pearson_correlation':float(pearsonr(old,new).statistic),'spearman_rank_correlation':float(spearmanr(old,new).statistic),'class_changes_at_0_5':changes}; print(json.dumps(pred_cmp,indent=2))
cv_delta=tu_mean-ch_mean; oof_delta=tu_oof_auc-ch_oof_auc; std_delta=tu_std-ch_std; decision='채택' if cv_delta>.0001 and oof_delta>.0001 and std_delta<=.005 else ('보류' if abs(cv_delta)<=.0001 or abs(oof_delta)<=.0001 else '제외'); print('decision:',decision)
summary={'experiment':'Titanic 16 - CatBoost RandomizedSearchCV','base_model':'Titanic 15','feature_changed':False,'features':FEATURES,'encoded_feature_count':int(e1.shape[1]),'n_iter':30,'scoring':'roc_auc','best_params':{k.replace('model__',''):v for k,v in search.best_params_.items()},'search_best_score':float(search.best_score_),'champion_cv_mean':float(ch_mean),'tuned_cv_mean':float(tu_mean),'champion_cv_std':float(ch_std),'tuned_cv_std':float(tu_std),'champion_oof_auc':float(ch_oof_auc),'tuned_oof_auc':float(tu_oof_auc),'champion_holdout':ch_hold,'tuned_holdout':tu_hold,'prediction_comparison':pred_cmp,'decision':decision,'submission_file':str(sub),'private_auc':'제출 후 입력','public_auc':'제출 후 입력'}
sp=Path('results/titanic_16_summary.json'); assert not sp.exists(); sp.write_text(json.dumps(summary,ensure_ascii=False,indent=2,default=float),encoding='utf-8'); print(sp)


{
  "mean_absolute_prediction_difference": 0.02629324939592306,
  "max_absolute_prediction_difference": 0.21297402281564404,
  "pearson_correlation": 0.9943338884426631,
  "spearman_rank_correlation": 0.9655511918534677,
  "class_changes_at_0_5": 5
}
decision: 채택
results\titanic_16_summary.json


# Titanic 16 최종 결과

Feature 변경: 없음 / Model: CatBoost → CatBoost / Hyperparameter tuning: RandomizedSearchCV / Search combinations: 30 / Scoring: ROC-AUC / CV: StratifiedKFold 5.

실행된 fixed CV Mean, OOF AUC, CV Std, holdout 비교에 따라 채택·보류·제외를 판단한다. Kaggle Private AUC와 Public AUC는 제출 후 입력으로 남긴다. RandomizedSearchCV best CV score는 parameter selection에 사용되었으므로 완전히 독립적인 성능 추정치가 아니다.
